In [48]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
import numpy as np
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [2]:
docx_loader = Docx2txtLoader('datasets/Introduction_to_Data_and_Data_Science_2.docx')
pages_docx = docx_loader.load()

In [3]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [('#', 'Course Title'),
                                                                ('##', 'Lecture Title')])

In [4]:
pages_md_split = md_splitter.split_text(pages_docx[0].page_content)

In [5]:
for i in pages_md_split:
    i.page_content = ' '.join(i.page_content.split())

In [6]:
char_splitter = CharacterTextSplitter(
    separator = '.',
    chunk_size = 500,
    chunk_overlap = 50
)

In [7]:
pages_char_split = char_splitter.split_documents(pages_md_split)
pages_char_split

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individu

In [8]:
%load_ext dotenv
%dotenv

In [9]:
embedding = OpenAIEmbeddings(model = "text-embedding-ada-002")

In [10]:
vector1 = embedding.embed_query(pages_char_split[3].page_content)
vector2 = embedding.embed_query(pages_char_split[5].page_content)
vector3 = embedding.embed_query(pages_char_split[18].page_content)

In [11]:
vector1

[0.003742626868188381,
 0.010484526865184307,
 0.010070833377540112,
 -0.03043227642774582,
 -0.01968919113278389,
 0.012074658647179604,
 -0.024239812046289444,
 -0.013018394820392132,
 0.010562093928456306,
 -0.027096876874566078,
 0.004783323034644127,
 0.01674162968993187,
 -0.013115353882312775,
 0.006341134663671255,
 -0.010898219421505928,
 -0.01657356694340706,
 0.03604298457503319,
 -0.0008976810495369136,
 0.020270947366952896,
 -0.02367098443210125,
 -0.04374801367521286,
 0.022494545206427574,
 -0.00870047602802515,
 -0.027407146990299225,
 -0.009999730624258518,
 0.0037006111815571785,
 0.010258288122713566,
 -0.0250284131616354,
 0.004815642721951008,
 -0.01821541041135788,
 0.009734708815813065,
 0.005672115832567215,
 -0.008002369664609432,
 0.0004969161236658692,
 -0.009055993519723415,
 0.0029475612100213766,
 0.011143849231302738,
 0.01587546057999134,
 0.015293705277144909,
 0.017633654177188873,
 0.01521613821387291,
 0.0009308087755925953,
 -0.03405208885669708,
 

In [12]:
len(vector1), len(vector2), len(vector3)

(1536, 1536, 1536)

In [13]:
np.dot(vector1, vector2), np.dot(vector1, vector3), np.dot(vector2, vector3)

(np.float64(0.8791284497943928),
 np.float64(0.80002358287471),
 np.float64(0.7934993700101873))

In [14]:
np.linalg.norm(vector1), np.linalg.norm(vector2), np.linalg.norm(vector3)

(np.float64(0.9999999518969219),
 np.float64(0.9999999432048748),
 np.float64(0.9999999688261215))

In [15]:
len(pages_char_split)

20

In [16]:
vectorstore = Chroma.from_documents(documents = pages_char_split,
                                    embedding = embedding,
                                    persist_directory = './intro-to-ds-lectures')

In [17]:
vectorstore_from_directory = Chroma(persist_directory = './intro-to-ds-lectures',
                                   embedding_function = embedding)

C:\Users\SmAsHeR\AppData\Local\Temp\ipykernel_6900\1124944897.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_from_directory = Chroma(persist_directory = './intro-to-ds-lectures',


In [18]:
vectorstore_from_directory.get()

{'ids': ['0c92ef23-25f0-49fe-8c52-fb0358f8a6ee',
  '0de4f13e-d51c-4230-8a57-2da38438f21a',
  'f0f4b178-fa81-4491-ab70-6e91c2620a6b',
  '9fd53439-0aee-4c6a-9cc6-c8d522febad9',
  '52481e57-3c8f-4889-85c0-c96be60a74cf',
  'c9db5a4b-0b10-4b79-a1fa-793b4b732572',
  '8172ab2d-a234-4e9a-8634-641944155715',
  'f3aaafa6-5a48-46b5-b8db-c985f7559576',
  'ecee458d-1c6c-41e4-a98f-1a770aebb81c',
  'edbab497-ba96-49fc-8ca4-9ef4d0bbfa5a',
  '58a46144-ea1d-489a-b32a-378c0ff7783f',
  '99341606-d9c7-4953-aafb-c41a83831922',
  '4b7e3940-b7ff-4b2d-821e-0c22c83ec56b',
  'aabdddac-e811-46e5-9d4c-c4c50b87b82f',
  '6d21a117-b07c-48d5-bfaf-b6db41c71ffa',
  'a3ca18e3-ecc4-41bf-8abb-cd12fecb11b3',
  '42d5ca8b-e513-42b1-9044-0472fe189409',
  '86079641-1a93-4cd7-a3db-5387e4c276a5',
  '850bb541-d33f-476b-b2dd-16d582ed5220',
  'e371fd3c-2d84-4762-b640-81ab789e57e6'],
 'embeddings': None,
 'documents': ['Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the simi

In [19]:
vectorstore_from_directory.get(ids = '0c92ef23-25f0-49fe-8c52-fb0358f8a6ee',
                              include = ['embeddings'])

{'ids': ['0c92ef23-25f0-49fe-8c52-fb0358f8a6ee'],
 'embeddings': array([[ 0.00478017, -0.01535145,  0.02508651, ...,  0.02121745,
         -0.01364157, -0.00687695]], shape=(1, 1536)),
 'documents': None,
 'uris': None,
 'included': ['embeddings'],
 'data': None,
 'metadatas': None}

In [20]:
pages_char_split[0]

Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis')

In [21]:
added_document = Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis')

In [22]:
vectorstore_from_directory.add_documents([added_document])

['7c57220f-0d89-472f-949d-2d732ad675f8']

In [23]:
vectorstore_from_directory.get('7c57220f-0d89-472f-949d-2d732ad675f8')

{'ids': ['7c57220f-0d89-472f-949d-2d732ad675f8'],
 'embeddings': None,
 'documents': ['Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Lecture Title': 'Analysis vs Analytics',
   'Course Title': 'Introduction to Data and Data Science'}]}

In [24]:
pages_char_split[19]

Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!')

In [25]:
updated_document = Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!')

In [26]:
vectorstore_from_directory.update_document(document_id = '7c57220f-0d89-472f-949d-2d732ad675f8',
                                           document = updated_document )

In [27]:
vectorstore_from_directory.get('7c57220f-0d89-472f-949d-2d732ad675f8')

{'ids': ['7c57220f-0d89-472f-949d-2d732ad675f8'],
 'embeddings': None,
 'documents': ['Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need',
   'Course Title': 'Introduction to Data and Data Science'}]}

In [28]:
vectorstore_from_directory.delete('7c57220f-0d89-472f-949d-2d732ad675f8')

In [29]:
vectorstore_from_directory.get('7c57220f-0d89-472f-949d-2d732ad675f8')

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [30]:
vectorstore = Chroma(persist_directory = './intro-to-ds-lectures',
                    embedding_function = embedding)

In [31]:
added_document = Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execute the same action')

In [32]:
vectorstore.add_documents([added_document])

['194eaf69-44b5-41aa-9317-dcccdf3a2623']

In [33]:
question = 'What programming languages do data scientists use?'

In [34]:
retrieved_docs = vectorstore.similarity_search(query = question, k = 5)

In [35]:
retrieved_docs

[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right! Let’s finish off with machine learning. When it comes to machine learning, we often deal with big data'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. Apart from R, Python, and MATLAB, other, faster languages are used

In [36]:
for i in retrieved_docs:
    print(f"Page Content: {i.page_content}\n---------\n Lecture Title: {i.metadata['Lecture Title']}\n")

Page Content: What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right! Let’s finish off with machine learning. When it comes to machine learning, we often deal with big data
---------
 Lecture Title: Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. Apart from R, Python, and MATLAB, other, faster languages are used like Java, JavaScript, C, C++, and Scala. Cool. What we said may be wonderful, but that’s not all! By using one or more programming languages, people create application software or, as they are sometimes called, software solutions, that are adjusted for 

In [37]:
question = 'What software do data scientists use?'

In [38]:
retrieved_docs = vectorstore.similarity_search(query = question, k = 3)

In [39]:
for i in retrieved_docs:
    print(f"Page Content : {i.page_content}\n---------------\nLecture Title : {i.metadata['Lecture Title']}\n")

Page Content : As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
---------------
Lecture Title : Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content : Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever 

In [40]:
retrieved_docs = vectorstore.max_marginal_relevance_search(query = question,
                                                           k = 3,
                                                           lambda_mult = 1,
                                                           filter = {"Lecture Title" : "Programming Languages & Software Employed in Data Science - All the Tools You Need"})

In [41]:
for i in retrieved_docs:
    print(f"Page Content : {i.page_content}\n----------\nLecture Title : {i.metadata['Lecture Title']}\n")

Page Content : As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
----------
Lecture Title : Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content : Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you n

In [42]:
len(vectorstore.get()['documents'])

21

In [43]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                    search_kwargs = {'k' : 3,
                                                     'lambda_mult' : 0.7})

In [44]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000028B1EE8FB50>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.7})

In [45]:
question = 'What softwares do data scientists use?'

In [46]:
retrieved_docs = retriever.invoke(question)

In [47]:
for i in retrieved_docs:
    print(f"Page Content : {i.page_content}\n----------\nLecture Title : {i.metadata['Lecture Title']}\n")

Page Content : As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
----------
Lecture Title : Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content : It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop distributes the computational tasks on multiple computers which is basically the way to handle big data nowadays. Power BI, SaS, Qlik, and especially Tableau are top-notch examples of software designed for business intelligence visualizations
----------
Lecture Ti

In [49]:
TEMPLATE = ''' 
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of response,specify the name of lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

In [50]:
prompt_template = PromptTemplate.from_template(TEMPLATE)

In [51]:
chat = ChatOpenAI(model = 'gpt-4',
                 seed = 365,
                 max_tokens = 250)

In [52]:
question = 'What software do data scientists use?'

In [53]:
chain = {'context' : retriever,
         'question' : RunnablePassthrough()}

In [54]:
chain.invoke(question)

AttributeError: 'dict' object has no attribute 'invoke'

In [55]:
chain = RunnableParallel({'context' : retriever,
                          'question' : RunnablePassthrough()})

In [56]:
chain.invoke(question)

{'context': [Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'),
  Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop

In [57]:
chain = {'context' : retriever, 
        'question' : RunnablePassthrough()} | prompt_template

In [58]:
chain.invoke(question)

StringPromptValue(text=" \nAnswer the following question:\nWhat software do data scientists use?\n\nTo answer the question, use only the following context:\n[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='It

In [59]:
print(" \nAnswer the following question:\nWhat software do data scientists use?\n\nTo answer the question, use only the following context:\n[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop distributes the computational tasks on multiple computers which is basically the way to handle big data nowadays. Power BI, SaS, Qlik, and especially Tableau are top-notch examples of software designed for business intelligence visualizations'), Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!')]\n\nAt the end of response,specify the name of lecture this context is taken from in the format:\nResources: *Lecture Title*\nwhere *Lecture Title* should be substituted with the title of all resource lectures.\n")

 
Answer the following question:
What software do data scientists use?

To answer the question, use only the following context:
[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='It’s actually a software framew

In [60]:
chain = ({'context' : retriever,
         'question' : RunnablePassthrough()}
        | prompt_template
        | chat
        | StrOutputParser())

In [61]:
chain.invoke(question)

'Data scientists use a variety of tools and software. Some of the most popular tools for data manipulation and integration within multiple data and data science software platforms are programming languages like R and Python. These languages are not only suitable for mathematical and statistical computations but are also adaptable and can solve a wide variety of business and data-related problems. On the other hand, a software framework known to handle the complexity of big data and its computational intensity is Hadoop. Hadoop can distribute computational tasks on multiple computers. Additionally, Power BI, SaS, Qlik, and Tableau are examples of software designed specifically for business intelligence visualizations.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need'

In [62]:
print('Data scientists use a variety of tools and software. Some of the most popular tools for data manipulation and integration within multiple data and data science software platforms are programming languages like R and Python. These languages are not only suitable for mathematical and statistical computations but are also adaptable and can solve a wide variety of business and data-related problems. On the other hand, a software framework known to handle the complexity of big data and its computational intensity is Hadoop. Hadoop can distribute computational tasks on multiple computers. Additionally, Power BI, SaS, Qlik, and Tableau are examples of software designed specifically for business intelligence visualizations.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need')

Data scientists use a variety of tools and software. Some of the most popular tools for data manipulation and integration within multiple data and data science software platforms are programming languages like R and Python. These languages are not only suitable for mathematical and statistical computations but are also adaptable and can solve a wide variety of business and data-related problems. On the other hand, a software framework known to handle the complexity of big data and its computational intensity is Hadoop. Hadoop can distribute computational tasks on multiple computers. Additionally, Power BI, SaS, Qlik, and Tableau are examples of software designed specifically for business intelligence visualizations.

Resources: Programming Languages & Software Employed in Data Science - All the Tools You Need
